# SDSS with `astroquery` — Cone Searches, Spectra & Images

**Task 4 (part b) of the DR19 tutorial list: the astroquery SDSS page.**

`astroquery.sdss` lets you query SDSS and pull down spectra and images directly from Python,
without going through the web tools. This notebook walks through the patterns you'll actually
reuse: a positional cross-match (cone search), region searches, downloading spectra/images,
and the built-in spectral templates.

> **Note:** every cell here talks to the live SDSS servers, so you need an internet
> connection to run them. If a query hangs or returns stale results, jump to the
> Troubleshooting section at the bottom and clear the cache.

### Default data release
By default `astroquery.sdss` targets **DR17** (the final SDSS-IV release: optical + IR spectra,
datacubes, stellar-library spectra and images). You can point any query at a different release
with the `data_release=` keyword.

In [ ]:
from astroquery.sdss import SDSS
from astropy import coordinates as coords
import astropy.units as u

print('Using astroquery; default queries hit DR17 unless data_release is set.')

## 1. Getting started — an individual cross-ID (cone search)

Start from a position found in some other survey and search a small circle around it for SDSS
optical counterparts. The `spectro=True` keyword restricts matches to objects that have
**spectroscopy**, not just photometry.

In [ ]:
pos = coords.SkyCoord('0h8m05.63s +14d50m23.3s', frame='icrs')
xid = SDSS.query_region(pos, radius='5 arcsec', spectro=True)
print(xid)

The result is an `astropy.Table`. Each row is a matched SDSS object; useful columns include
`ra`, `dec`, `objid`, `specobjid`, `run`, `camcol`, `field`, `plate`, `mjd` and `fiberID` —
the identifiers you need to pull the actual data in the download step below.

## 2. Searching regions and multiple objects

`query_region` accepts a single coordinate **or a list/Column of coordinates**, and the *shape*
of the search depends on which keyword you pass:

- **`radius`** → a circle around each point (a cone search). In this mode `query_region` is
  equivalent to `query_crossid`, and SDSS enforces a hard **3 arcmin** ceiling on `radius`.
- **`width`** (optionally with **`height`**) → a rectangle in RA/Dec centred on each point.
  The rectangle is the intuitive "this range of RA, that range of Dec"; it does **not** apply the
  cos(δ) correction, so near the poles the box is really a trapezoid on the sky. It *does* handle
  RA wrap-around, so searches near RA = 0 still cover a sensible region.

You must supply **either** `radius` **or** `width` — passing neither or both raises an error.

In [ ]:
# Cone search around several positions at once
positions = coords.SkyCoord(
    ra=[2.0234, 10.9, 150.1]*u.deg,
    dec=[14.84, -0.5, 2.3]*u.deg,
    frame='icrs')
multi = SDSS.query_region(positions, radius=10*u.arcsec, spectro=True)
print(multi)

In [ ]:
# Rectangular search: 1 arcmin wide in RA, 2 arcmin tall in Dec
rect = SDSS.query_region(pos, width=1*u.arcmin, height=2*u.arcmin)
print(rect)

## 3. Downloading data

Once you have an `xid` table from `query_region`, you already hold everything needed to fetch the
matching spectra and images. Pass the table straight in via `matches=`.

In [ ]:
sp = SDSS.get_spectra(matches=xid)          # list of HDUList, one per row in xid
im = SDSS.get_images(matches=xid, band='g')  # list of HDUList, one per row in xid

print('spectra returned :', len(sp))
print('images returned  :', len(im))

Both return **lists of `HDUList` objects**, one entry per object in `xid`.

Heads-up on images: SDSS image downloads return the **entire plate/frame**, not a postage stamp.
If you want a cutout centred on your target you'll need to slice it out yourself afterwards
(using the WCS in the frame header).

In [ ]:
# Peek at the first downloaded spectrum
hdul = sp[0]
hdul.info()

## 4. Spectral templates

SDSS ships a set of template spectra you can download directly.

> **Warning (from the docs):** these templates come from the SDSS-I/II pipeline (DR7 and earlier).
> The SDSS-III/IV pipelines (DR8+) use different templates, so don't mix them with modern spectra
> without care.

List what's available, then grab one by name.

In [ ]:
print(SDSS.AVAILABLE_TEMPLATES)

In [ ]:
template = SDSS.get_spectral_template('qso')
print('templates returned:', len(template))   # 'galaxy' returns 3; most return 1
template[0].info()

## 5. Troubleshooting

If queries keep failing, or you're getting bad / out-of-date results, clear the local cache:

In [ ]:
SDSS.clear_cache()
print('cache cleared')

If `clear_cache` doesn't exist, upgrade astroquery — it was added in 0.4.7. Check your version:

In [ ]:
import astroquery
print('astroquery', astroquery.__version__)

### Where this fits in your project
For the void-finding / galaxy-clustering side, the workhorse is a **scripted SQL query** rather
than these positional lookups — see the companion notebook `sdss_sql_casjobs_tutorial.ipynb`,
which uses `SDSS.query_sql` to pull a galaxy sample you can turn into 3-D positions. Use the cone
searches here when you're checking specific targets or cross-matching against another catalog.

---
*Reference: astroquery.sdss documentation. Explanations paraphrased; code adapted from the docs.*